# Testing and performance

Based on the performance on validation set, we choose kNN for score prediction and XGBoost for cost prediction and the routing policy defined before. We now run this policy on the test data set. 

In [1]:
# Load the saved policy and test prompts

from pathlib import Path
import json
import sys
import importlib

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import load_npz

PROJECT_DIR = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "1_data_preparation").is_dir()
    ),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the project.")

DATA_DIR = (
    PROJECT_DIR
    / "1_data_preparation/data/processed/char_tfidf_50k"
)

POLICY_PATH = (
    PROJECT_DIR / "artifacts" / "router_knn_xgb" / "policy.json"
)

with POLICY_PATH.open() as file:
    policy_config = json.load(file)

selected_lambda = float(policy_config["lambda"])
REFERENCE_MODEL = policy_config["reference_model"]
model_names = policy_config["model_order"]
validation_target = policy_config["validation_quality_target"]

assert policy_config["cost_units"] == "dollars_per_prompt"

test_questions = pd.read_pickle(DATA_DIR / "test_questions.pkl")

print("Test prompts:", len(test_questions))
print("Fixed lambda:", selected_lambda)
print("Saved validation quality target:", validation_target)

Test prompts: 7300
Fixed lambda: 2.7142066911362983
Saved validation quality target: 0.78785968758564


In [2]:
# Predict scores using kNN
module_dir = str(PROJECT_DIR / "2a_Score_training_calibration")

if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

import score_model_knn as knn_module

importlib.reload(knn_module)

SCORE_MODEL_DIR = PROJECT_DIR / policy_config["score_artifact"]

with (SCORE_MODEL_DIR / "metadata.json").open() as file:
    score_metadata = json.load(file)

score_knn = joblib.load(
    SCORE_MODEL_DIR / score_metadata["model_file"]
)

score_vectorizer = joblib.load(
    SCORE_MODEL_DIR / score_metadata["vectorizer_file"]
)

X_test_score = score_vectorizer.transform(
    test_questions["question_text"]
)

assert X_test_score.shape[1] == score_metadata["n_features"]
assert set(score_metadata["model_names"]) == set(model_names)

selected_k = score_metadata["k"]

score_predictions = knn_module.predict_score_grid(
    score_knn,
    X_test_score,
    k_values=[selected_k],
    batch_size=score_metadata["batch_size"],
)[selected_k]

q_hat_test = pd.DataFrame(
    score_predictions,
    index=test_questions.index,
    columns=score_metadata["model_names"],
).loc[:, model_names]

print("Score predictions:", q_hat_test.shape)

Score predictions: (7300, 11)


In [3]:
# Import reusable cost-model loading and dollar prediction helpers.
module_dir = str(PROJECT_DIR / "2b_cost_training_calibration")

if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

import cost_model_xgb as cost_module

importlib.reload(cost_module)

# Predict test costs using the artifact referenced by the saved policy.
COST_MODEL_DIR = PROJECT_DIR / policy_config["cost_artifact"]

cost_models, cost_metadata = cost_module.load_cost_models(COST_MODEL_DIR)

X_test_cost = load_npz(DATA_DIR / "X_test.npz")

assert X_test_cost.shape[0] == len(test_questions)
assert set(cost_models) == set(model_names)

c_hat_test = cost_module.predict_costs(
    X_test_cost,
    cost_models,
    cost_metadata,
    index=test_questions.index,
).loc[:, model_names]

assert q_hat_test.index.equals(c_hat_test.index)
assert q_hat_test.columns.equals(c_hat_test.columns)
assert np.isfinite(q_hat_test.to_numpy()).all()
assert np.isfinite(c_hat_test.to_numpy()).all()
assert q_hat_test.ge(0).all().all()
assert q_hat_test.le(1).all().all()
assert c_hat_test.ge(0).all().all()

print("Cost predictions:", c_hat_test.shape)

Cost predictions: (7300, 11)


In [4]:
# Apply the fixed routing policy
utility = (
    q_hat_test.to_numpy()
    - selected_lambda * c_hat_test.to_numpy()
)

chosen_columns = utility.argmax(axis=1)
rows = np.arange(len(test_questions))

test_routing_details = test_questions[
    ["sample_id", "eval_name", "question_group"]
].copy()

test_routing_details["chosen_model"] = (
    q_hat_test.columns.to_numpy()[chosen_columns]
)

test_routing_details["predicted_score"] = (
    q_hat_test.to_numpy()[rows, chosen_columns]
)

test_routing_details["predicted_cost"] = (
    c_hat_test.to_numpy()[rows, chosen_columns]
)

print("Routing decisions made:", len(test_routing_details))

Routing decisions made: 7300


In [5]:
# Load outcomes and test performance

test_scores = pd.read_pickle(
    DATA_DIR / "test_scores.pkl"
).astype(float)

test_costs = pd.read_pickle(
    DATA_DIR / "test_costs.pkl"
).astype(float)

assert test_scores.index.equals(test_questions.index)
assert test_costs.index.equals(test_questions.index)

test_scores = test_scores.loc[:, model_names]
test_costs = test_costs.loc[:, model_names]

assert np.isfinite(test_scores.to_numpy()).all()
assert np.isfinite(test_costs.to_numpy()).all()

test_routing_details["observed_score"] = (
    test_scores.to_numpy()[rows, chosen_columns]
)

test_routing_details["observed_cost"] = (
    test_costs.to_numpy()[rows, chosen_columns]
)

test_routing_details["gpt4_score"] = test_scores[REFERENCE_MODEL]
test_routing_details["gpt4_cost"] = test_costs[REFERENCE_MODEL]

reference_score = test_scores[REFERENCE_MODEL].mean()
reference_cost = test_costs[REFERENCE_MODEL].mean()

assert reference_cost > 0

test_comparison = pd.DataFrame({
    "observed_mean_score": [
        reference_score,
        test_routing_details["observed_score"].mean(),
    ],
    "cost_per_1000_prompts": [
        1000 * reference_cost,
        1000 * test_routing_details["observed_cost"].mean(),
    ],
}, index=[
    "Always GPT-4",
    "kNN + XGBoost router",
])

test_comparison["quality_gap_vs_test_gpt4"] = (
    test_comparison["observed_mean_score"] - reference_score
)

test_comparison["gap_vs_saved_validation_target"] = (
    test_comparison["observed_mean_score"] - validation_target
)

test_comparison["savings_per_1000_prompts"] = (
    1000 * reference_cost
    - test_comparison["cost_per_1000_prompts"]
)

test_comparison["cost_savings_pct"] = 100 * (
    1
    - test_comparison["cost_per_1000_prompts"]
    / (1000 * reference_cost)
)

display(test_comparison.round(6))

,observed_mean_score,cost_per_1000_prompts,quality_gap_vs_test_gpt4,gap_vs_saved_validation_target,savings_per_1000_prompts,cost_savings_pct
Always GPT-4,0.779795,3.280085,0.00000,-0.008065,0.000000,0.000000
kNN + XGBoost router,0.780534,2.044736,0.00074,-0.007325,1.235349,37.662092


In [6]:

from scipy.stats import bootstrap

bootstrap_rows = test_routing_details.copy()

assert bootstrap_rows["question_group"].notna().all()

bootstrap_rows["score_gap"] = (
    bootstrap_rows["observed_score"] - bootstrap_rows["gpt4_score"]
)

group_totals = (
    bootstrap_rows.groupby("question_group")
    .agg(
        prompts=("observed_score", "size"),
        score_gap_sum=("score_gap", "sum"),
        router_cost_sum=("observed_cost", "sum"),
        gpt4_cost_sum=("gpt4_cost", "sum"),
    )
)

bootstrap_data = tuple(
    group_totals[column].to_numpy(dtype=float)
    for column in [
        "prompts",
        "score_gap_sum",
        "router_cost_sum",
        "gpt4_cost_sum",
    ]
)


def comparison_statistics(
    counts, score_gaps, router_costs, gpt4_costs, axis=-1
):
    n = counts.sum(axis=axis)
    r_cost = router_costs.sum(axis=axis)
    g_cost = gpt4_costs.sum(axis=axis)

    return np.stack([
        score_gaps.sum(axis=axis) / n,
        1000 * (g_cost - r_cost) / n,
        100 * (1 - r_cost / g_cost),
    ])


bootstrap_result = bootstrap(
    bootstrap_data,
    comparison_statistics,
    paired=True,
    vectorized=True,
    n_resamples=5000,
    batch=128,
    confidence_level=0.95,
    method="basic",
    rng=np.random.default_rng(42),
)

test_uncertainty = pd.DataFrame({
    "estimate": comparison_statistics(*bootstrap_data),
    "CI_95_lower": bootstrap_result.confidence_interval.low,
    "CI_95_upper": bootstrap_result.confidence_interval.high,
}, index=[
    "Quality gap versus GPT-4",
    "Dollars saved per 1,000 prompts",
    "Cost savings (%)",
])

display(test_uncertainty.round(6))

,estimate,CI_95_lower,CI_95_upper
Quality gap versus GPT-4,0.000740,-0.002973,0.004541
"Dollars saved per 1,000 prompts",1.235349,1.175772,1.291616
Cost savings (%),37.662092,36.393698,38.913973
